# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/haroonrana330/flyrank-ml/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

### My baseline rule

I rank pages for refresh review using four signals: search visibility, freshness risk, position opportunity, and content depth. Higher visibility, older pages, stronger page-one opportunity, and thinner content increase the baseline refresh score.

The rule is a decision-support baseline rather than a final decision. It helps identify pages that deserve human review first.

### Reason codes

- `stale_visible_page` — the page has not been updated recently and still has meaningful impressions.
- `declining_with_demand` — the page is trending down while still receiving meaningful impressions.
- `thin_visible_page` — the page has relatively low word count but still receives meaningful impressions.
- `page_one_decay_risk` — the page is ranking on page one but is relatively old.
- `low_ctr_visible_page` — the page has meaningful impressions, a visible search position, and low CTR.
- `low_engagement_visible_page` — the page has sessions but weak engagement or scroll behaviour.
- `general_refresh_review` — none of the specific conditions above were triggered.

In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import numpy as np

url = "https://raw.githubusercontent.com/haroonrana330/flyrank-ml/main/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(url)

print("Rows:", len(df))
print("Columns:", len(df.columns))
df.head()

Rows: 30000
Columns: 44


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

### Baseline scoring method

The baseline score combines four transparent components:

- Visibility score: 40%
- Freshness risk score: 30%
- Position opportunity score: 25%
- Content depth gap score: 5%

The final score is between 0 and 1. Pages with higher scores are placed earlier in the review queue.

In [16]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import numpy as np
from pathlib import Path

# Make sure df exists
if "df" not in globals():
    url = "https://raw.githubusercontent.com/haroonrana330/flyrank-ml/main/data/raw/content_refresh_anonymized.csv"
    df = pd.read_csv(url)

# Make required numeric columns safe
numeric_columns = [
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "content_age_days",
    "days_since_last_update",
    "word_count",
    "avg_position",
    "ctr",
    "engagement_rate",
    "scroll_rate"
]

for col in numeric_columns:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)

# Create label from observed trend
if "trend_direction" in df.columns:
    df["is_declining_label"] = (
        df["trend_direction"].astype(str).str.lower() == "down"
    ).astype(int)
else:
    df["is_declining_label"] = 0

# Percentile rank helper
def percentile_rank(series):
    return series.rank(pct=True).fillna(0)

# Normalization helper
def normalize(series):
    minimum = series.min()
    maximum = series.max()
    if maximum == minimum:
        return pd.Series(0.0, index=series.index)
    return (series - minimum) / (maximum - minimum)

# Build the baseline scores
df["visibility_score"] = percentile_rank(
    np.log1p(df["impressions_90d"])
)

df["freshness_risk_score"] = percentile_rank(
    df["days_since_last_update"]
)

df["position_opportunity_score"] = (
    (1 - normalize(df["avg_position"].clip(lower=1, upper=50)))
    * df["visibility_score"]
    * (df["avg_position"] > 0).astype(int)
)

df["depth_gap_score"] = (
    1 - percentile_rank(df["word_count"])
) * df["visibility_score"]

df["baseline_refresh_score"] = (
    0.40 * df["visibility_score"]
    + 0.30 * df["freshness_risk_score"]
    + 0.25 * df["position_opportunity_score"]
    + 0.05 * df["depth_gap_score"]
).clip(0, 1)

# Create reason codes if Section 1 did not already create them
def make_reason_codes(row):
    reasons = []

    if row["days_since_last_update"] >= 180 and row["impressions_90d"] >= 500:
        reasons.append("stale_visible_page")

    if (
        str(row.get("trend_direction", "")).lower() == "down"
        and row["impressions_90d"] >= 100
    ):
        reasons.append("declining_with_demand")

    if (
        row["word_count"] > 0
        and row["word_count"] < 1200
        and row["impressions_90d"] >= 250
    ):
        reasons.append("thin_visible_page")

    if (
        row["avg_position"] > 0
        and row["avg_position"] <= 10
        and row["content_age_days"] >= 180
    ):
        reasons.append("page_one_decay_risk")

    if (
        row["impressions_90d"] >= 500
        and 0 < row["avg_position"] <= 20
        and row["ctr"] < 0.5
    ):
        reasons.append("low_ctr_visible_page")

    if (
        row["sessions_90d"] >= 30
        and (
            (row["engagement_rate"] > 0 and row["engagement_rate"] < 30)
            or
            (row["scroll_rate"] > 0 and row["scroll_rate"] < 30)
        )
    ):
        reasons.append("low_engagement_visible_page")

    if not reasons:
        reasons.append("general_refresh_review")

    return "|".join(reasons)

df["reason_codes"] = df.apply(make_reason_codes, axis=1)

# Suggested action
def get_action(reason_codes):
    reasons = set(str(reason_codes).split("|"))

    if "thin_visible_page" in reasons:
        return "expand_and_refresh"

    if "low_ctr_visible_page" in reasons:
        return "refresh_and_review_ctr"

    if (
        "stale_visible_page" in reasons
        or "declining_with_demand" in reasons
    ):
        return "refresh"

    return "monitor"

df["suggested_action_baseline"] = df["reason_codes"].apply(get_action)

# Rank everything
df["baseline_rank"] = (
    df["baseline_refresh_score"]
    .rank(method="first", ascending=False)
    .astype(int)
)

# Create output
output_columns = [
    "content_id",
    "client_id",
    "baseline_rank",
    "baseline_refresh_score",
    "visibility_score",
    "freshness_risk_score",
    "position_opportunity_score",
    "depth_gap_score",
    "reason_codes",
    "suggested_action_baseline",
    "is_declining_label",
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "avg_position",
    "ctr",
    "engagement_rate",
    "scroll_rate",
    "content_age_days",
    "days_since_last_update",
    "word_count",
    "trend_direction"
]

output_columns = [
    col for col in output_columns
    if col in df.columns
]

ranked_queue = (
    df[output_columns]
    .sort_values("baseline_rank")
    .reset_index(drop=True)
)

# Save exactly where the assignment asks
output_path = Path("work/outputs/baseline_action_score.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)

ranked_queue.to_csv(output_path, index=False)

print("Baseline queue created successfully.")
print("Rows:", len(ranked_queue))
print("Output:", output_path)
print()
print(ranked_queue.head(10))

Baseline queue created successfully.
Rows: 30000
Output: work/outputs/baseline_action_score.csv

             content_id          client_id  baseline_rank  \
0  content_9532f197bbc8  client_4e07408562              1   
1  content_4d1fe5b32dc2  client_19581e27de              2   
2  content_07f2e7a6f38a  client_19581e27de              3   
3  content_e5ae436f9a16  client_4e07408562              4   
4  content_3430a8b94511  client_19581e27de              5   
5  content_cbd93118300b  client_19581e27de              6   
6  content_9c195417f6ef  client_19581e27de              7   
7  content_ba2acb4ebd04  client_19581e27de              8   
8  content_79b25654070a  client_19581e27de              9   
9  content_adddad39251c  client_19581e27de             10   

   baseline_refresh_score  visibility_score  freshness_risk_score  \
0                0.941189          0.999633                0.8432   
1                0.934889          0.994167                0.8432   
2                0.93408

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

### Top-20 review

I reviewed the highest-ranked pages from the baseline queue. The baseline score determines priority, while the reason code explains why a page was selected.

The confidence note describes why the recommendation appears reasonable from the observed data. The "what would make it wrong" note records an important limitation that should be checked before taking action.

In [17]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd

# Load the ranked queue created in Section 2
queue_path = "work/outputs/baseline_action_score.csv"
ranked_queue = pd.read_csv(queue_path)

top20 = ranked_queue.head(20).copy()

def confidence_note(score):
    if score >= 0.75:
        return "High"
    elif score >= 0.50:
        return "Medium"
    else:
        return "Low"

def what_would_make_it_wrong(reason):
    reason = str(reason)

    if "stale_visible_page" in reason:
        return "The page may already have been updated recently."

    if "declining_with_demand" in reason:
        return "The decline may be temporary or noisy."

    if "thin_visible_page" in reason:
        return "The short content may already satisfy the search need."

    if "page_one_decay_risk" in reason:
        return "The page may still perform well despite its age."

    if "low_ctr_visible_page" in reason:
        return "Low CTR may be caused by the query or SERP layout."

    if "low_engagement_visible_page" in reason:
        return "Low engagement may reflect the page's search intent."

    return "The available signals may not indicate a real refresh need."

top20_review = pd.DataFrame({
    "rank": top20["baseline_rank"],
    "content_id": top20["content_id"],
    "action": top20["suggested_action_baseline"],
    "reason_code": top20["reason_codes"],
    "confidence": top20["baseline_refresh_score"].apply(confidence_note),
    "what_would_make_it_wrong": top20["reason_codes"].apply(
        what_would_make_it_wrong
    )
})

print("TOP-20 REVIEW")
print("=" * 80)
print(top20_review.to_string(index=False))

TOP-20 REVIEW
 rank           content_id                 action                                                                                reason_code confidence                         what_would_make_it_wrong
    1 content_9532f197bbc8                refresh                      declining_with_demand|page_one_decay_risk|low_engagement_visible_page       High           The decline may be temporary or noisy.
    2 content_4d1fe5b32dc2                monitor                                            page_one_decay_risk|low_engagement_visible_page       High The page may still perform well despite its age.
    3 content_07f2e7a6f38a                monitor                                            page_one_decay_risk|low_engagement_visible_page       High The page may still perform well despite its age.
    4 content_e5ae436f9a16 refresh_and_review_ctr                       page_one_decay_risk|low_ctr_visible_page|low_engagement_visible_page       High The page may still perform wel

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

### Weak picks and leakage check

Some high-ranked pages may still be weak recommendations. A high baseline score does not prove that a page should actually be changed. For example, an old page can remain useful, low CTR can be caused by search intent, and thin content can still satisfy the user's need.

The baseline is therefore a prioritisation tool for human review, not an automatic publishing decision.

For leakage, the score uses observed page-level signals available in the prepared feature vector. I did not use client names, private queries, future-window labels, or product flags to calculate the baseline score.

In [18]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd

queue_path = "work/outputs/baseline_action_score.csv"
ranked_queue = pd.read_csv(queue_path)

print("WEAK PICKS")
print("=" * 80)

# Show the lowest-scoring items among the top 20
top20 = ranked_queue.head(20)

weak_picks = top20.tail(5)[[
    "baseline_rank",
    "content_id",
    "baseline_refresh_score",
    "reason_codes",
    "suggested_action_baseline"
]]

print(weak_picks.to_string(index=False))

print()
print("WHY THESE PICKS MAY BE WRONG")
print("=" * 80)

for _, row in weak_picks.iterrows():
    print(
        f"Rank {row['baseline_rank']}: "
        f"The score is relatively weaker, so the available signals may "
        f"not be enough to justify a refresh decision."
    )

print()
print("LEAKAGE CHECK")
print("=" * 80)

# These are outcome/future-related fields that should not be used
# as inputs to the baseline score.
forbidden_inputs = [
    "is_declining_label",
    "trend_direction",
    "trend_pct"
]

print("Outcome/future-related fields present in output:")
for col in forbidden_inputs:
    if col in ranked_queue.columns:
        print("-", col)

print()
print("The baseline score itself uses:")
print("- impressions_90d")
print("- days_since_last_update")
print("- avg_position")
print("- word_count")

print()
print("Leakage check completed.")
print("The outcome fields are kept for review/evaluation, not used to calculate the score.")

WEAK PICKS
 baseline_rank           content_id  baseline_refresh_score                                                          reason_codes suggested_action_baseline
            16 content_2c2606c5d176                0.930059 declining_with_demand|page_one_decay_risk|low_engagement_visible_page                   refresh
            17 content_9351f948bf45                0.930058                       page_one_decay_risk|low_engagement_visible_page                   monitor
            18 content_37106924f264                0.929529                       page_one_decay_risk|low_engagement_visible_page                   monitor
            19 content_f4c93868660b                0.929302                       page_one_decay_risk|low_engagement_visible_page                   monitor
            20 content_8818fd6d967f                0.929007 declining_with_demand|page_one_decay_risk|low_engagement_visible_page                   refresh

WHY THESE PICKS MAY BE WRONG
Rank 16: The score is r

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.